# Images photometry

After an observation is done, a common need is to reduce and extract fluxes from raw FITS images.

In this tutorial you will learn how to process a complete night of raw data from any telescope by building a data reduction [Sequence](prose.Sequence) with prose.

In [1]:
!ls ~/.prose/

In [2]:
!cat ~/.prose/tcs.telescope

In [3]:
# date = '241210'
date = '250104'
fits_folder = f"/data/MuSCAT2/{date}"

In [4]:
## Showing obslog
inst='muscat2'
for i in range(4):
    print('\n=== CCD{0} ==='.format(i))
    !perl /ut3/muscat/obslog/show_obslog_summary.pl $inst $date $i

In [5]:
!ls /data/MuSCAT2/$date | wc -l

In [6]:
# target = 'TOI06982.01'
target = 'TOI06909.01'

In [7]:
from astropy.io import fits
from pathlib import Path
from tqdm import tqdm

objs = {}
darks = {}
flats = {}
files = list(Path(fits_folder).glob('*.fits'))
for f in tqdm(files):
    h = fits.getheader(f)
    flt = h['FILTER']
    if h['OBJECT']==target:
        if flt not in objs:
            objs[flt]=[]
        else:
            objs[flt].append(f)
    elif h['OBJECT']=='DARK':
        if flt not in darks:
            darks[flt]=[]
        else:
            darks[flt].append(f)
    elif h['OBJECT']=='FLAT':
        if flt not in flats:
            flats[flt]=[]
        else:
            flats[flt].append(f)

In [8]:
objs = {b: sorted(objs[b]) for b in objs}
flats = {b: sorted(flats[b]) for b in flats}
darks = {b: sorted(darks[b]) for b in darks}

here prose simulated comparison stars, their fluxes over time and some systematic noises.

## Explore FITS

The first thing we want to do is to see what is contained within our folder. For that we instantiate a [FitsManager](prose.FitsManager) object on our folder to describe its content

In [9]:
!ls $fits_folder | wc -l

## Extracting photometry

### Using a reference

In order to perform the photometric extraction of stars fluxes on all images, we will select a reference image from which we will extract:
- The stars positions, then reused and detected on all images (after being aligned to the reference)
- The global full-width at half-maximum (fwhm) of the PSF, to scale the apertures used in the [aperture photometry block](prose.blocks.AperturePhotometry)

In [10]:
from prose import FITSImage

band = 'i'
N = len(objs[band])
ref = FITSImage(objs[band][N//2])

In [11]:
from prose import Sequence, blocks

calibration = Sequence(
    [
        blocks.Calibration(darks=darks[band], flats=flats[band]),
        blocks.Trim(),
        blocks.PointSourceDetection(n=15),  # stars detection
        blocks.Cutouts(51),  # making stars cutouts
        blocks.MedianEPSF(),  # building PSF
        blocks.Moffat2D(),  # modeling PSF
    ]
)

calibration.run(ref, show_progress=True)
ref.show(contrast=0.1)
# ref.sources.plot()

In [12]:
import matplotlib.pyplot as plt

plt.figure(None, (10, 4))

# PSF building
plt.subplot(1, 2, 1, title="Median PSF")
plt.imshow(ref.epsf.data, origin="lower")

# PSF modeling
params = ref.epsf.params
model = ref.epsf.model

plt.subplot(1, 2, 2, title="Moffat2D model")
plt.imshow(model(params), origin="lower")
_ = plt.text(
    1,
    1,
    f"$\sigma_x$: {params['sigma_x']:.2f} pix\n$\sigma_y$: {params['sigma_y']:.2f} pix",
    c="w",
)

```{tip}
You can use a Gaia query to define which stars you want the photometry from (e.g. if too faint to be detected here)
```

### Aperture photometry

We can now extract the photometry of these stars

In [13]:
import numpy as np
radii = np.arange(4, 12, 2) #pix
# radii = np.array([10])

photometry = Sequence(
    [
        *calibration,  # calibration
        blocks.ComputeTransformTwirl(ref),  # compute alignment
        blocks.AlignReferenceSources(ref),  # alignment
        blocks.CentroidQuadratic(),  # centroiding
        blocks.AperturePhotometry(radii=radii),  # aperture photometry
        blocks.AnnulusBackground(),  # local background estimate
        blocks.GetFluxes(
            "fwhm",
            airmass=lambda im: im.header["AIRMASS"],
            dx=lambda im: im.transform.translation[0],
            dy=lambda im: im.transform.translation[1],),
    ]
)

# photometry.run(objs[band][:N//10])
photometry.run(objs[band])

In [ ]:
photometry.__dict__.keys()

In [ ]:
photometry.last_image.show()

## The [Fluxes](prose.Fluxes) object

The [GetFluxes](prose.blocks.GetFluxes) block provide a way to retrieve the fluxes computed by the [AperturePhotometry](prose.blocks.AperturePhotometry) minus the background estimated with [AnnulusBackground](prose.blocks.AnnulusBackground)

In [ ]:
from prose import Fluxes

fluxes: Fluxes = photometry[-1].fluxes

In [ ]:
import matplotlib.pyplot as plt

# picking target
raw_fluxes.target = 0

# good practice
raw_fluxes = raw_fluxes.sigma_clipping_data(bkg=3)

# differential photometry
diff_fluxes = raw_fluxes.autodiff()
diff_fluxes.plot()
# diff_fluxes.bin(0.005, True).errorbar()

# plt.ylim(0.98, 1.02)
plt.xlabel("time")
plt.ylabel("diff. flux")
plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt

raw_fluxes.target = 6

# a bit of cleaning
nan_stars = np.any(np.isnan(raw_fluxes.fluxes), axis=(0, 2)) # stars with nan fluxes
fluxes = raw_fluxes.mask_stars(~nan_stars) # mask nans stars
fluxes = raw_fluxes.sigma_clipping_data(bkg=3, fwhm=3) # sigma clipping

# differential photometry
diff = fluxes.autodiff()

# plotting
ax = plt.subplot(xlabel="time (JD)", ylabel="diff. flux", 
                 # ylim=(0.94, 1.06)
                )
diff.plot()
diff.bin(5 / 60 / 24, estimate_error=True).errorbar()

In [ ]:
plt.figure(None, (5, 7))
ax = plt.subplot(xlabel="time (JD)", ylabel="diff. flux (abritrary units)")


# plotting only the first five comparisons
for j, i in enumerate([diff.target, *diff.comparisons[0:5]]):
    y = diff.fluxes[diff.aperture, i].copy()
    y = (y - np.mean(y)) / np.std(y) + 8 * j
    plt.text(
        diff.time.max(), np.mean(y) + 4, i if i != diff.target else "target", ha="right"
    )
    plt.plot(diff.time, y, ".", c="0.8" if i != diff.target else "k")

In [ ]:
plt.figure(None, (5, 7))
ax = plt.subplot(xlabel="time (JD)", ylabel="scaled signals (abritrary units)")

for i, name in enumerate(["flux", "fwhm", "airmass", "bkg", "dx", "dy"]):
    y = diff.df[name].copy()
    y = (y - np.mean(y)) / np.std(y) + 8 * i
    plt.text(diff.time.max(), np.mean(y) + 4, name, ha="right")
    plt.plot(diff.time, y, ".", c="0.8" if name != "flux" else "k")